# Transaktions-Klassifizierung: Memory + ML

Kategorisiert Transaktionen anhand des Händlers. **Die Reihenfolge entspricht der Architektur:**
1. **Memory (Stufe 1)** schlägt bekannte Händler exakt nach — der Hauptweg.
2. **ML-Fallback (Stufe 3)** greift nur für unbekannte Händler.

Läuft **lokal** (kein Cloud-LLM → Daten bleiben auf dem Gerät).

## Imports & Setup

Alle Bibliotheken laden, den Projekt-Parser (`backend/parsers`) auf den Pfad legen und das geheime `PSEUDO_SALT` aus der `.env` holen.

In [ ]:
import os
import sys
import json
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments

# backend/ auf den Pfad legen, damit die Projekt-Module importierbar sind.
# Nach oben suchen statt fixem parents[n] -> überlebt das Verschieben des Notebooks.
BACKEND = next(p / "backend" for p in Path.cwd().parents if (p / "backend").is_dir())
sys.path.insert(0, str(BACKEND))
from parsers import DKB
from anonymize import pseudonymize_df, suspected_person_payees
from payee import to_merchant_key, effective_key

DATA_DIR = BACKEND.parent / "data"

# Geheimes Salt aus .env laden (nicht committen; per .gitignore ausgeschlossen)
load_dotenv(BACKEND.parent / ".env")
SALT = os.environ["PSEUDO_SALT"]

## 1. Kontoauszug laden & anonymisieren

Den synthetischen Auszug durch den DKB-Parser schicken, die Labels (`Kategorie`/`Unterkategorie`) anhängen und **sofort** anonymisieren (IBAN schwärzen, Privatpersonen pseudonymisieren) — bevor etwas angezeigt wird.

In [11]:
# Große synthetische Basis (2.640 Zeilen, ~80 Beispiele/Unterkat., 451 distinkte Händler).
# Erzeugt mit generate_synthetic_data.py -> genug Händler-Vielfalt für einen ehrlichen
# händler-gruppierten Test. Für einen echten Auszug hier einfach den Pfad tauschen.
STATEMENT = Path.cwd() / "dkb_synth_large.csv"

df1 = DKB.load(str(STATEMENT))

# Labels (Kategorie/Unterkategorie) übernehmen -- Kategorien sind keine PII.
# Kopfzeile dynamisch (dieselbe Erkennung wie DKB.load) -> kein fixes skiprows.
raw = pd.read_csv(STATEMENT, sep=";", encoding="utf-8", skiprows=DKB.find_header_row(str(STATEMENT)))
df1["category"] = raw["Kategorie"].values
df1["subcategory"] = raw["Unterkategorie"].values
del raw  # Rohdaten nicht unnötig im Speicher lassen

# Anonymisieren: bei ECHTEN Auszügen Privatkontakte in KNOWN_PERSONS eintragen.
# suspected_person_payees(df1)          # einmal ausführen, um Kandidaten zu sichten
KNOWN_PERSONS = []                       # z. B. ["Max Mustermann", "Anna Schmidt"]
df1 = pseudonymize_df(
    df1,
    salt=SALT,
    known_persons=KNOWN_PERSONS,
    redact_columns=("account",),         # ggf. "description" ergänzen, wenn Freitext PII enthält
)

display(df1.head())
print(df1["category"].value_counts(dropna=False))

,transaction_id,booking_date,value_date,amount,currency,payee,description,source,transaction_type,category,account,subcategory
0,f9883d9164de7e43bcbe21a2c6144d1c,2026-06-15,2026-06-15,-26.39,EUR,1und1 DSL,Rechnung 06/2026 Kd-Nr. 059784,DKB,Ausgang,8. fixed costs,[redacted],internet
1,f84414c85f4d06d83ec240103ff18ad5,2026-04-13,2026-04-13,-27.41,EUR,PayPal Europe S.a.r.l. et Cie S.C.A,"9946418398416 PP.4855.PP . CITY TAXI, Ihr Eink...",DKB,Ausgang,3. mobility,[redacted],taxi
2,856e39b4b4995cb5f6af0416b7457974,2025-10-06,2025-10-06,-25.01,EUR,Deutsche Glasfaser,Rechnung 10/2025 Kd-Nr. 951522345,DKB,Ausgang,8. fixed costs,[redacted],internet
3,de80476ed5213c416f98cb62c30d188f,2026-03-16,2026-03-16,-22.75,EUR,Spargelhof.Klein.0047/Düsseldorf,SPARGELHOF SAGT DANKE. 2026-03-16T22:18:34 KFN...,DKB,Ausgang,1. groceries,[redacted],farmers market
4,8b9c18e82f056021402dea959086c040,2025-10-05,2025-10-05,-207.46,EUR,Rossmann Babywelt,Lastschrift 10/2025 Ref 170598445,DKB,Ausgang,4. family,[redacted],family


category
2. lifestyle&fun    640
8. fixed costs      560
7. abos             560
3. mobility         320
1. groceries        160
6. healthcare       160
5. education        160
4. family            80
Name: count, dtype: int64


## 2. Händler-Key bilden

`effective_key` verdichtet jeden payee zu einer stabilen Händler-Kennung — die Grundlage für **beide** Stufen.
Schreibvarianten (`REWE Markt Kiel` ↔ `REWE.Markt…/Kiel`) ergeben denselben Key; bei Zahlungsdienstleistern (PayPal) wird der echte Händler aus dem Verwendungszweck gezogen.

In [12]:
# payee -> effektiver Händler-Key (von Memory UND ML-Fallback genutzt).
# Bei Zahlungsdienstleistern (PayPal) den echten Händler aus der description ziehen,
# sonst den payee bereinigen.
df1["merchant_key"] = df1.apply(lambda r: effective_key(r["payee"], r["description"]), axis=1)
df1[["payee", "merchant_key", "category"]].head(10)

,payee,merchant_key,category
0,1und1 DSL,unddsl,8. fixed costs
1,PayPal Europe S.a.r.l. et Cie S.C.A,citytaxi,3. mobility
2,Deutsche Glasfaser,deutscheglasfaser,8. fixed costs
3,Spargelhof.Klein.0047/Düsseldorf,spargelhofklein,1. groceries
4,Rossmann Babywelt,rossmannbabywelt,4. family
5,wilhelm.tel,wilhelmtel,8. fixed costs
6,KVB Köln,kvb,3. mobility
7,PayPal Europe S.a.r.l. et Cie S.C.A,taxigenossenschaft,3. mobility
8,PayPal Europe S.a.r.l. et Cie S.C.A,adlerapotheke,6. healthcare
9,Hotel.Astoria.124/Köln,hotelastoria,2. lifestyle&fun


## 3. Memory (Stufe 1) — der primäre Kategorisierer

Bekannte Händler werden **exakt** nachgeschlagen (`merchant_key → category`): kein ML, kein Raten, 100 % korrekt.
Nur wirklich unbekannte Händler fallen auf `"unsicher"`. Weil sich Händler in echten Auszügen ständig wiederholen, deckt das Memory den Großteil ab.

In [13]:
# Memory: persistent in memory.json -> wächst über Läufe und Sitzungen hinweg
MEMORY_PATH = Path.cwd() / "memory.json"


def load_memory() -> dict:
    return json.loads(MEMORY_PATH.read_text()) if MEMORY_PATH.exists() else {}


def save_memory(mem: dict) -> None:
    MEMORY_PATH.write_text(json.dumps(mem, ensure_ascii=False, indent=2, sort_keys=True))


def build_memory(df) -> dict:
    """merchant_key -> häufigste category aus gelabelten Zeilen."""
    labeled = df.dropna(subset=["category"])
    return {key: grp["category"].mode().iloc[0]
            for key, grp in labeled.groupby("merchant_key")}


# vorhandenes Memory laden und mit den Labels aus diesem Auszug ergänzen
memory = load_memory()
memory.update(build_memory(df1))
save_memory(memory)
print(f"{len(memory)} Händler im Memory (memory.json)")

505 Händler im Memory (memory.json)


In [14]:
# Stufe 1: exakter Memory-Treffer über den effektiven Key, sonst "unsicher".
# description wird für Zahlungsdienstleister (PayPal) gebraucht.
def categorize(payee: str, description: str = "") -> str:
    return memory.get(effective_key(payee, description), "unsicher")


def remember(payee: str, kategorie: str, description: str = "") -> str:
    """Bestätigte Kategorie ins Memory schreiben UND persistieren -> künftig exakter Treffer."""
    key = effective_key(payee, description)
    memory[key] = kategorie
    save_memory(memory)
    return key


# bekannt -> korrekt;  PayPal -> echter Händler aus der description;  unbekannt -> "unsicher"
proben = [
    ("REWE Markt Kiel", ""),
    ("PayPal Europe S.a.r.l. et Cie S.C.A", "PP.4855.PP . NETFLIX, Ihr Einkauf bei NETFLIX"),
    ("PayPal Europe S.a.r.l. et Cie S.C.A", "PP.4855.PP . FREENOW, Ihr Einkauf bei FREENOW"),
    ("Unbekannt GmbH", ""),
]
for p, d in proben:
    print(f"{p[:26]:26s} -> [{effective_key(p, d):12s}] -> {categorize(p, d)}")

REWE Markt Kiel            -> [rewe        ] -> 1. groceries
PayPal Europe S.a.r.l. et  -> [netflix     ] -> 7. abos
PayPal Europe S.a.r.l. et  -> [freenow     ] -> 3. mobility
Unbekannt GmbH             -> [unbekannt   ] -> unsicher


**Ehrliche Messung:** Memory aus „Vergangenheit" bauen, auf „Gegenwart" anwenden — so misst man die reale Abdeckung.

In [15]:
# Ehrliche Messung: Memory aus "Vergangenheit" bauen, auf "Gegenwart" anwenden.
# Zufälliger ZEILEN-Split (nicht händler-gruppiert!) -> Händler wiederholen sich,
# genau wie in echten monatlichen Auszügen.
labeled = df1.dropna(subset=["category"])
past, present = train_test_split(labeled, test_size=0.25, random_state=42)

mem_past = build_memory(past)
present = present.copy()
present["pred"] = present.apply(
    lambda r: mem_past.get(effective_key(r["payee"], r["description"]), "unsicher"), axis=1
)

beantwortet = present["pred"] != "unsicher"
abdeckung = beantwortet.mean()
korrekt = (present.loc[beantwortet, "pred"] == present.loc[beantwortet, "category"]).mean()

print(f"Abdeckung (bekannte Händler): {abdeckung:.0%}")
print(f"davon korrekt:               {korrekt:.0%}")
print(f"'unsicher' (neue Händler):    {1 - abdeckung:.0%}")

Abdeckung (bekannte Händler): 100%
davon korrekt:               100%
'unsicher' (neue Händler):    0%


### Dazulernen mit `remember(...)`

Einen bisher unbekannten Händler bestätigen — das Memory merkt es sich **dauerhaft** in `memory.json` (überlebt Kernel-Neustart und neue Sitzungen). So wächst die Abdeckung mit der Zeit Richtung 100 %.

In [16]:
print("vorher:  Shell Tankstelle ->", categorize("Shell Tankstelle"))
remember("Shell Tankstelle", "3. mobility")   # einmal bestätigen
print("nachher: Shell Tankstelle ->", categorize("Shell Tankstelle"))
print("-> steht jetzt in memory.json und überlebt einen Neustart")

vorher:  Shell Tankstelle -> 3. mobility
nachher: Shell Tankstelle -> 3. mobility
-> steht jetzt in memory.json und überlebt einen Neustart


## 4. ML-Fallback (Stufe 3) — nur für den „unsicher"-Rest

Für unbekannte Händler ein kleines Modell (SetFit). Mit der größeren Datenbasis (viele *verschiedene* Händler pro Kategorie) lässt sich die Generalisierung jetzt **ehrlich** messen: händler-gruppierter Split (Test = nur nie gesehene Händler), Macro-F1 statt nackter Accuracy, plus eine Konfidenzschwelle, ab der eine Buchung an den Nutzer geht statt geraten zu werden.

**Daten für den Fallback.** `merchant_key` als Text, `category` als Label. Zwei Schritte:
1. **Händler-gruppierter Split** — ein Händler ist entweder in Train **oder** Test, nie in beiden → misst echte Generalisierung auf unbekannte Händler.
2. **Few-shot-Sampling** aus dem Train-Pool — SetFit ist für *wenige* Beispiele gebaut; dank der neuen Vielfalt ziehen wir pro Klasse wenige, aber *verschiedene* Händler (statt 50× denselben).

> Hinweis: `LABEL_COL` steht auf `"category"` (8 Klassen, robust). Für den späteren Produktiv-Klassifikator auf `"subcategory"` umstellen (33 Klassen) — braucht die jetzt vorhandene Händler-Vielfalt.

In [17]:
# Label-Ebene: "category" (8 Klassen, robust) oder "subcategory" (33 Klassen, Produktiv-Ziel)
LABEL_COL = "category"

data = (
    df1[["merchant_key", LABEL_COL]]
    .dropna()
    .drop_duplicates()                       # ein Händler zählt einmal je Label (kein Wiederhol-Bias)
    .rename(columns={"merchant_key": "text", LABEL_COL: "label"})
)

# 1) HÄNDLER-gruppierter Split: Test enthält NUR Händler, die im Training nie vorkamen.
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(data, groups=data["text"]))
train_pool = data.iloc[train_idx]
test_df = data.iloc[test_idx]

# 2) Few-shot: pro Klasse wenige, aber VERSCHIEDENE Händler ziehen (SetFits Stärke).
#    Über Gruppen iterieren + Indizes sammeln -> versionssicher (neuere pandas schließen
#    die Gruppierungsspalte in groupby().apply() aus -> 'label' ginge sonst verloren).
N_SHOTS = 12
sampled_idx = []
for _, g in train_pool.groupby("label"):
    sampled_idx.extend(g.sample(min(len(g), N_SHOTS), random_state=42).index)
train_df = train_pool.loc[sampled_idx]

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
test_ds = Dataset.from_pandas(test_df, preserve_index=False)

assert set(train_ds.column_names) >= {"text", "label"}, train_ds.column_names
print(f"Train: {len(train_ds)} (<= {N_SHOTS}/Klasse)   Test: {len(test_ds)}   |   "
      f"distinkte Händler: {data['text'].nunique()}   Klassen: {data['label'].nunique()}")
print("Test = nur UNBEKANNTE Händler -> ehrliche Generalisierungs-Messung")

Train: 95 (<= 12/Klasse)   Test: 118   |   distinkte Händler: 472   Klassen: 8
Test = nur UNBEKANNTE Händler -> ehrliche Generalisierungs-Messung


**SetFit trainieren** und auf dem Test (nur unbekannte Händler) ehrlich messen: Accuracy, **Macro-F1** (gewichtet alle Klassen gleich — seltene zählen genauso wie `supermarket`) und eine **Konfidenzschwelle** (Abdeckung ↔ Genauigkeit), um Stufe 3 zu kalibrieren.

In [18]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

args = TrainingArguments(batch_size=16, num_epochs=1)   # few-shot -> auf CPU wenige Minuten
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds)
trainer.train()

# --- Ehrliche Metriken auf ausschließlich UNBEKANNTEN Händlern ---
X_test = test_ds["text"]
y_true = np.array(test_ds["label"])
y_pred = np.array(model.predict(X_test))
proba = np.asarray(model.predict_proba(X_test))
conf = proba.max(axis=1)                 # Konfidenz = höchste Klassenwahrscheinlichkeit

print(f"Accuracy : {accuracy_score(y_true, y_pred):.1%}")
print(f"Macro-F1 : {f1_score(y_true, y_pred, average='macro'):.1%}   "
      f"(alle Klassen gleich gewichtet)")

# --- Konfidenzschwelle kalibrieren: ab wann fällt eine Buchung auf 'unsicher' (-> Nutzer)? ---
print("\nSchwelle | Abdeckung | Genauigkeit(abgedeckt)")
for t in (0.0, 0.4, 0.5, 0.6, 0.7, 0.8):
    take = conf >= t
    cov = take.mean()
    acc = accuracy_score(y_true[take], y_pred[take]) if take.any() else float("nan")
    print(f"   {t:>3.1f}   |   {cov:5.0%}   |        {acc:5.0%}")
print("-> Schwelle so wählen, dass 'Genauigkeit(abgedeckt)' hoch ist; der Rest geht an den Nutzer.")

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Map: 100%|██████████| 95/95 [00:00<00:00, 19261.32 examples/s]
***** Running training *****
  Num unique pairs = 7896
  Batch size = 16
  Num epochs = 1
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.
/Users/clarabrilke/ML projects/moneta/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.226200
50,0.239800
100,0.189500
150,0.140800
200,0.092700
250,0.070100
300,0.048600
350,0.040500
400,0.022800
450,0.020900


Accuracy : 35.6%
Macro-F1 : 37.9%   (alle Klassen gleich gewichtet)

Schwelle | Abdeckung | Genauigkeit(abgedeckt)
   0.0   |    100%   |          36%
   0.4   |     92%   |          38%
   0.5   |     78%   |          40%
   0.6   |     57%   |          43%
   0.7   |     48%   |          47%
   0.8   |     38%   |          51%
-> Schwelle so wählen, dass 'Genauigkeit(abgedeckt)' hoch ist; der Rest geht an den Nutzer.


**Auf neue payees anwenden** — bereinigt (`to_merchant_key`) wie im Training.

In [19]:
# Wichtig: neue payees GENAUSO bereinigen wie im Training (to_merchant_key)
beispiele = ["REWE Markt Kiel", "Deutsche Bahn", "Netflix", "Shell Tankstelle", "Fitnessstudio McFit"]
keys = [to_merchant_key(b) for b in beispiele]
for text, key, pred in zip(beispiele, keys, model.predict(keys)):
    print(f"{text:22s} -> [{key:20s}] -> {pred}")

REWE Markt Kiel        -> [rewe                ] -> 1. groceries
Deutsche Bahn          -> [deutschebahn        ] -> 3. mobility
Netflix                -> [netflix             ] -> 7. abos
Shell Tankstelle       -> [shelltankstelle     ] -> 1. groceries
Fitnessstudio McFit    -> [fitnessstudiomcfit  ] -> 6. healthcare
